In [1]:
import numpy as np
vec_file = "DG_50hZ_8reps_8dir_2sT.vec"

# Load vec file
vec = np.loadtxt(vec_file)[
    1:, :
]  # Remove first line as it is not a trigger
vec_header = np.loadtxt(vec_file, max_rows=1)
print(f"\nSelected vec file : {vec_file}\n")
print(f"Vec file length : {vec.shape[0]}")


Selected vec file : DG_50hZ_8reps_8dir_2sT.vec

Vec file length : 19200


In [2]:
len(vec)

19200

In [3]:
k = 0
j = 0
for elem in vec:
    if 0.0 in elem:
        k+=1
    for nb in elem:
        if nb != 0.0:
            j+=1
print(k, j)

19200 16000


In [4]:
16000/(20*8)

100.0

In [5]:
19200/600, 8*4

(32.0, 32)

In [6]:
vec_std = vec.copy()

N_DIRECTIONS = 8           # number of drifting-grating directions
DETECTIONS_PER_REP = 5     # each repetition produces this many detection events (one per grating-phase marker)
FRAMES_PER_GRATING = 100   # one 2 s grating period at 50 Hz = 100 frames; also the half-window labelled around each event

rep_max_length = 1000

# Count detection events seen so far, per direction.
# Use an integer counter (not a float accumulator like += 0.2): summing 0.2 ten times
# gives 1.9999999999999998, so int(...) would truncate to 1 instead of 2 and shift a
# repetition boundary by one grating period. Integer division avoids that entirely.
detection_count = {direction: 0 for direction in range(1, N_DIRECTIONS + 1)}

for k, elem in enumerate(vec):
    for grating_nb in range(0, N_DIRECTIONS):
        frame_nb = grating_nb * FRAMES_PER_GRATING + 1
        if frame_nb in elem:
            direction = grating_nb + 1
            rep = detection_count[direction] // DETECTIONS_PER_REP
            for i in range(k - FRAMES_PER_GRATING, k + FRAMES_PER_GRATING):
                vec_std[i][-1] = direction * rep_max_length * 10 + rep
            detection_count[direction] += 1

In [7]:
detection_count

{1: 20, 2: 20, 3: 20, 4: 20, 5: 20, 6: 20, 7: 20, 8: 20}

In [8]:
vec_std

array([[    0.,     0.,     0.,     0., 10000.],
       [    0.,     0.,     0.,     0., 10000.],
       [    0.,     0.,     0.,     0., 10000.],
       ...,
       [    0.,    98.,     0.,     0., 10003.],
       [    0.,    99.,     0.,     0., 10003.],
       [    0.,   100.,     0.,     0., 10003.]])

In [9]:
full_std_vec = np.concatenate((vec_header.reshape(1, -1), vec_std), axis=0)

np.savetxt("DG_50hZ_8reps_8dir_2sT_std.vec", full_std_vec, fmt="%1.f")